# Corn Yield Prediction Model Training Pipeline

This notebook documents the end-to-end machine learning pipeline for predicting corn yield in the Anuradhapura district. The objective is to build a high-precision regression model that can forecast harvest weight based on environmental, soil, and management factors.

### Key Stages:
1. **Data Ingestion**: Loading regional agricultural datasets.
2. **Preprocessing**: Cleaning categorical labels and handling outliers.
3. **Feature Engineering**: Aligning the dataset with the primary predictor variables.
4. **Pipeline Construction**: Embedding encoders and imputers for production consistency.
5. **Model Benchmarking**: Comparing RandomForest, Gradient Boosting, and XGBoost.
6. **Evaluation**: Assessing performance via RMSE, MAE, and R² scores.
7. **Serialization**: Exporting the best-performing model for backend integration.

In [ ]:
# Install specific versions of ML libraries to ensure binary compatibility 
# between the training environment and the production backend.
!pip install -U scikit-learn==1.7.2 xgboost==3.2.0 joblib==1.5.3 shap==0.48.0 -q

In [ ]:
# =============================================================================
# 1. LIBRARY IMPORTS
# =============================================================================
import numpy as np
import pandas as pd
import joblib
import warnings

# Scikit-learn utilities for model selection and preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Regression metrics for quantifying prediction error
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Ensemble estimators known for high performance on tabular data
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Suppress non-critical warnings to keep output clean for reports
warnings.filterwarnings("ignore")

## 1. Dataset Loading

We use **Pandas** for data ingestion due to its efficient handling of structured CSV data. The dataset represents regional agricultural records from **Anuradhapura**, a key corn-producing area. Focusing on a specific district allows the model to learn localized climate and soil interactions that might be lost in a national-level dataset.

* **Target Variable**: `yield_kg_per_acre` (The value we aim to predict).
* **Input Features**: 9 key environmental and management variables.

In [ ]:
data_path = "anuradhapura_corn_yield_dataset.csv"
df = pd.read_csv(data_path)

# Display structural metadata to verify successful loading
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

## 2. Data Cleaning & Feature Selection

Preprocessing is critical because machine learning models are highly sensitive to inconsistent labels and outliers. "Garbage in, garbage out"—bad data leads to poor generalization and misleading predictions.

### Key Actions:
1. **Required Columns**: Filtering the dataset to match the features collected by the mobile app.
2. **String Normalization**: Ensuring labels like 'anuradhapura' and 'Anuradhapura' are treated as the same category.
3. **Pest Level Mapping**: Converting numeric indices back to categorical strings (`None`, `Low`, etc.) to improve interpretability.
4. **Outlier Clipping**: Using the **Interquartile Range (IQR)** method to bound extreme values that could skew the model's loss function.

In [ ]:
# Filter for features present in the production inference schema
required_columns = [
    "district", "variety", "soil_type", "irrigation_type", "pest_disease_level",
    "farm_size_acres", "seasonal_rainfall_mm", "fertilizer_kg_per_acre",
    "previous_yield_kg_per_acre", "yield_kg_per_acre"
]

df = df[required_columns].copy()

# Normalize categorical strings to prevent duplicate categories due to case/whitespace
cat_cols = ["district", "variety", "soil_type", "irrigation_type", "pest_disease_level"]
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

# Standardize specific agricultural labels
df["variety"] = df["variety"].replace({"Hybrid A": "Hybrid_A", "Hybrid B": "Hybrid_B", "OPV Local": "OPV_Local"})
df["soil_type"] = df["soil_type"].replace({"Loamy": "Loam", "loam": "Loam", "clay": "Clay", "sandy": "Sandy"})
df["pest_disease_level"] = df["pest_disease_level"].replace({
    "0": "None", "1": "Low", "2": "Medium", "3": "High",
    0: "None", 1: "Low", 2: "Medium", 3: "High"
})

# Convert numeric columns and handle non-numeric noise with 'coerce'
num_cols = ["farm_size_acres", "seasonal_rainfall_mm", "fertilizer_kg_per_acre", "previous_yield_kg_per_acre"]
target_col = "yield_kg_per_acre"
for col in num_cols + [target_col]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Define the Outlier Clipping function using IQR (Interquartile Range)
def clip_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return series.clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)

for col in num_cols + [target_col]:
    df[col] = clip_outliers_iqr(df[col])

print("\nPreprocessed Target Summary:")
print(df[target_col].describe())

## 3. Feature Engineering Context

The selected features represent the core drivers of corn productivity:
* **Rainfall & Irrigation**: Critical for water stress management in semi-arid Anuradhapura.
* **Fertilizer**: Direct input for nutrient availability.
* **Pest Level**: Acts as a yield reducer; high incidence can cause massive losses.
* **Previous Yield**: Captures historical soil quality and farmer management skill.

## 4. Train / Test Split

We utilize an **80/20 split** to ensure the model is trained on a substantial majority of the data while reserving 20% for evaluation on "unseen" data. This is essential for detecting **overfitting** (where the model memorizes training data but fails in the real world).

* `random_state=42`: Ensures the split is reproducible for future audits and peer review.

In [ ]:
X = df.drop(columns=[target_col])
y = df[target_col]

# Split data into training and holdout validation sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")

## 5. ColumnTransformer & Pipeline Construction

Modern ML requires embedding preprocessing steps directly into the **Pipeline**. This ensures consistency—the same transformations (imputation, encoding) are applied exactly the same way during both training and production inference.

* **SimpleImputer**: Fills missing values with the median (numeric) or mode (categorical) to prevent crashing.
* **OneHotEncoder**: Converts categorical text into numeric sparse vectors, as ML algorithms only operate on numbers.

In [ ]:
# Categorical pipeline: Impute missing -> One-Hot Encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Numeric pipeline: Impute missing with median to remain robust to outliers
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Assemble the column-specific transformations into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, cat_cols),
        ("num", numeric_transformer, num_cols),
    ]
)

## 6. Model Training (Candidate Benchmarking)

We evaluate three high-performance ensemble models:
1. **RandomForest**: Uses 'bagging' to reduce variance. Conceptually, it builds many trees and averages them to prevent any single tree from skewing the result.
2. **GradientBoosting**: Uses 'boosting' to reduce bias. It builds trees sequentially, where each new tree focuses on correcting the errors of the previous ones.
3. **XGBoost**: A highly optimized version of Gradient Boosting that includes regularization, making it extremely efficient and accurate for structured data.

In [ ]:
models = {
    "RandomForest": RandomForestRegressor(
        n_estimators=300, max_depth=10, min_samples_leaf=3, random_state=42, n_jobs=-1
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42
    ),
    "XGBoost": XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=4, subsample=0.9, random_state=42
    )
}

## 7. Model Evaluation & Benchmarking

We use four primary metrics to assess model quality:
* **MAE (Mean Absolute Error)**: Average magnitude of the errors (easy to interpret).
* **RMSE (Root Mean Squared Error)**: Penalizes larger errors more heavily; critical for agricultural yield where large misses can have high economic impact.
* **R² Score**: Indicates the proportion of variance explained by the model (Goal: > 0.90).
* **Cross-Validation (CV)**: Repeatedly trains and tests on different subsets of the data to ensure the performance is consistent and not a result of a lucky split.

In [ ]:
results = []
best_rmse = float("inf")
best_pipeline = None

cv = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    # Create a unified pipeline for each candidate
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train, y_train)
    
    # Predict on unseen test data
    preds = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2 = r2_score(y_test, preds)

    # Perform 5-fold cross-validation for reliability
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring="r2", n_jobs=-1)

    results.append({
        "Model": name, "MAE": mae, "RMSE": rmse, "R2_test": r2, "CV_R2_mean": cv_scores.mean()
    })
    
    print(f"{name} -> RMSE: {rmse:.2f}, R2: {r2:.4f}")

    # Track the best model based on the lowest prediction error (RMSE)
    if rmse < best_rmse:
        best_rmse = rmse
        best_pipeline = pipeline
        best_name = name

## 8. Best Model Selection

The best model is selected based on a combination of lowest **RMSE** and highest **CV mean R²**. This ensures we choose the model that not only performs best on the holdout set but also shows the most stability across multiple cross-validation folds.

In [ ]:
results_df = pd.DataFrame(results).sort_values(by="RMSE")
print("\nFull Model Comparison Table:")
print(results_df)

print(f"\nSELECTED MODEL: {best_name}")

## 9. Feature Importance Analysis

Tree-based models learn by identifying which features contribute most to reducing error (variance) during node splits. Unlike linear regressions with static weights, ensemble models use "Importance Scores" to show which variables dominate the decision path.

### Significance:
* High importance for **Rainfall** justifies the integration of Weather APIs in the frontend.
* Importance of **Variety** highlights the impact of genetic selection on harvest success.

In [ ]:
trained_model = best_pipeline.named_steps["model"]
ohe = best_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]

encoded_cat_names = ohe.get_feature_names_out(cat_cols)
feature_names = list(encoded_cat_names) + num_cols

if hasattr(trained_model, "feature_importances_"):
    fi = pd.DataFrame({"feature": feature_names, "importance": trained_model.feature_importances_})
    fi = fi.sort_values("importance", ascending=False)
    print("\nTop Factors Driving Predictions:")
    print(fi.head(10))

## 10. Model Export (Serialization)

We serialize the entire `Pipeline` using **Joblib**. This binary format preserves the fitted state of both the preprocessor and the estimator, allowing the FastAPI backend to load the artifact and perform live inference without retraining.

In [ ]:
model_path = "corn_yield_model_best.pkl"
joblib.dump(best_pipeline, model_path)
print(f"Successfully exported {best_name} to {model_path}")

## 11. Final Inference Test

Verifying that a manual sample correctly flows through the pipeline and produces a logical prediction.

In [ ]:
sample = pd.DataFrame([{
    "district": "Anuradhapura", "variety": "OPV_Local", "soil_type": "Loam",
    "irrigation_type": "Rainfed", "pest_disease_level": "High",
    "farm_size_acres": 0.5, "seasonal_rainfall_mm": 1000,
    "fertilizer_kg_per_acre": 130, "previous_yield_kg_per_acre": 1800
}])

prediction = best_pipeline.predict(sample)[0]
print(f"Predicted Yield for Sample: {prediction:.2f} kg/acre")

## 12. Environment & Dependency Audit

Critical for maintenance and deployment to ensure exact library version matching.

In [ ]:
import sklearn, xgboost
print(f"scikit-learn: {sklearn.__version__}")
print(f"xgboost:      {xgboost.__version__}")

## 13. Model Explainability (SHAP)

**SHAP (SHapley Additive exPlanations)** is a game-theoretic approach to explain the output of any machine learning model. In agriculture AI, explainability is as important as accuracy; a farmer needs to know *why* a model predicts a lower yield (e.g., is it due to insufficient rainfall or high pest incidence?).

* **SHAP Values**: Represent the contribution of each feature to the difference between the actual prediction and the average prediction.
* **Interpretation**: A positive SHAP value for rainfall means that the specific rainfall amount increased the predicted yield relative to the baseline.

In [ ]:
import shap

# 1. Initialize the SHAP explainer
# We use TreeExplainer as it is highly optimized for ensemble tree models like XGBoost/GBR
explainer = shap.TreeExplainer(best_pipeline.named_steps["model"])

# 2. Transform the test data using the pipeline's preprocessor
X_test_transformed = best_pipeline.named_steps["preprocessor"].transform(X_test)

# 3. Calculate SHAP values for the test set
shap_values = explainer.shap_values(X_test_transformed)

# 4. Visualize the global feature importance using a summary plot
print("\nSHAP Summary Plot (Global Impact):")
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names, plot_type="bar")